# 01 — EDA e coerência por rail

Objetivo: reproduzir a primeira leitura da base AML-FT, validando estrutura, qualidade e coerência por rail antes da criação de regras.

Este notebook foi desenhado para rodar a partir da raiz do repositório `aml-ft-case/`.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
TZ = "America/Sao_Paulo"
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "raw" / "AML/FT Transaction Monitoring Case Study INC (2).xlsx"
OUT_DIR = ROOT / "outputs" / "eda_day1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Root: {ROOT}")
print(f"Base existe: {DATA_PATH.exists()} | {DATA_PATH}")

## 1. Carregamento das abas

A primeira checagem é confirmar quais abas existem e se a estrutura corresponde ao que foi solicitado no case: transações, KYC, merchants, comportamento geográfico e dicionário.

In [ ]:
xl = pd.ExcelFile(DATA_PATH)
print(xl.sheet_names)

sheets = {name: xl.parse(name) for name in xl.sheet_names}
transactions = sheets.get("Transactions")
kyc = sheets.get("KYC_Profiles")
merchants = sheets.get("Merchants")
geo = sheets.get("GeoBehavior")
data_dictionary = sheets.get("Data_Dictionary")

summary = []
for name, df in sheets.items():
    summary.append({
        "sheet": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
    })
sheet_summary = pd.DataFrame(summary)
sheet_summary.to_csv(OUT_DIR / "01_sheet_summary.csv", index=False)
sheet_summary

## 2. Perfil rápido de colunas e nulos

Nulos não são necessariamente problema em AML. Em muitos casos, eles indicam que o campo não se aplica ao rail da transação. Mesmo assim, eu separo nulos reais de campos naturalmente não aplicáveis.

In [ ]:
def profile_dataframe(df: pd.DataFrame, name: str) -> pd.DataFrame:
    prof = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(t) for t in df.dtypes],
        "null_count": df.isna().sum().values,
        "null_pct": (df.isna().mean().values * 100).round(2),
        "nunique": df.nunique(dropna=True).values,
    })
    prof.to_csv(OUT_DIR / f"profile_{name}.csv", index=False)
    return prof

profile_transactions = profile_dataframe(transactions, "transactions")
profile_kyc = profile_dataframe(kyc, "kyc")
profile_merchants = profile_dataframe(merchants, "merchants")

profile_transactions.head(15)

## 3. Validações básicas de qualidade

Aqui eu valido pontos mínimos antes de seguir: IDs, duplicatas, timestamps, valores negativos e integridade entre transações e tabelas auxiliares.

In [ ]:
tx = transactions.copy()
tx["timestamp"] = pd.to_datetime(tx["timestamp"], errors="coerce")

quality_checks = {
    "transactions_rows": len(tx),
    "transaction_id_duplicates": int(tx["transaction_id"].duplicated().sum()) if "transaction_id" in tx else None,
    "invalid_timestamps": int(tx["timestamp"].isna().sum()),
    "amount_brl_le_zero": int((tx["amount_brl"] <= 0).sum()) if "amount_brl" in tx else None,
    "min_timestamp": tx["timestamp"].min(),
    "max_timestamp": tx["timestamp"].max(),
}

if {"customer_id"}.issubset(tx.columns) and {"customer_id"}.issubset(kyc.columns):
    quality_checks["tx_without_kyc"] = int((~tx["customer_id"].isin(kyc["customer_id"])).sum())

if {"merchant_id"}.issubset(tx.columns) and {"merchant_id"}.issubset(merchants.columns):
    quality_checks["tx_without_merchant"] = int((~tx["merchant_id"].isin(merchants["merchant_id"])).sum())

pd.DataFrame([quality_checks]).T.rename(columns={0: "value"})

## 4. Distribuição por rail

A análise por rail é essencial porque PIX, Card e Wire possuem campos, riscos e padrões diferentes.

In [ ]:
rail_col = "transaction_type"
rail_counts = tx[rail_col].value_counts(dropna=False).rename_axis("rail").reset_index(name="qtd")
rail_counts["pct"] = (rail_counts["qtd"] / len(tx) * 100).round(2)

amount_stats = tx.groupby(rail_col)["amount_brl"].agg(
    qtd="count",
    soma_brl="sum",
    media_brl="mean",
    mediana_brl="median",
    p95_brl=lambda s: s.quantile(0.95),
    max_brl="max",
).reset_index()

amount_stats.to_csv(OUT_DIR / "02_amount_stats_by_type.csv", index=False)
rail_counts

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.bar(rail_counts["rail"].astype(str), rail_counts["qtd"])
plt.title("Volume de transações por rail")
plt.xlabel("Rail")
plt.ylabel("Quantidade de transações")
plt.tight_layout()
plt.savefig(OUT_DIR / "chart_01_volume_por_rail.png", dpi=160)
plt.show()

plt.figure(figsize=(8, 4.5))
plt.bar(amount_stats[rail_col].astype(str), amount_stats["soma_brl"])
plt.title("Valor total movimentado por rail")
plt.xlabel("Rail")
plt.ylabel("Valor total em BRL")
plt.tight_layout()
plt.savefig(OUT_DIR / "chart_02_valor_total_por_rail.png", dpi=160)
plt.show()

## 5. Coerência por rail

Essa parte testa se os campos fazem sentido dentro de cada rail. Exemplo: PIX deve ter `pix_flow`; cartão e wire não precisam desse campo da mesma forma.

In [ ]:
checks = []

def add_check(name: str, condition_count: int, notes: str):
    checks.append({"check": name, "exceptions": int(condition_count), "notes": notes})

if "pix" in tx.columns and "transaction_type" in tx.columns:
    add_check("PIX com flag pix diferente de Yes", ((tx["transaction_type"] == "PIX") & (tx["pix"].astype(str) != "Yes")).sum(), "PIX deve estar marcado como pix=Yes")
if "pix_flow" in tx.columns:
    add_check("PIX sem pix_flow", ((tx["transaction_type"] == "PIX") & (tx["pix_flow"].isna() | (tx["pix_flow"].astype(str).str.lower() == "n/a"))).sum(), "PIX deve indicar cash_in ou cash_out")
if "auth_3ds" in tx.columns and "card_present" in tx.columns:
    add_check("Card e-commerce sem auth_3ds preenchido", ((tx["transaction_type"] == "Card") & (tx["card_present"].astype(str) == "No") & (tx["auth_3ds"].isna())).sum(), "E-commerce deve ser avaliado por 3DS/ECI")
if "receiver_country" in tx.columns:
    add_check("Wire sem receiver_country", ((tx["transaction_type"] == "Wire") & (tx["receiver_country"].isna())).sum(), "Wire cross-border depende de país de destino")

rail_checks = pd.DataFrame(checks)
rail_checks.to_csv(OUT_DIR / "03_rail_coherence_checks.csv", index=False)
rail_checks

## 6. Sinais AML iniciais

A EDA também levanta sinais iniciais que serão usados como insumo para regras: PEP, sanções, país de alto risco, MCC de risco, device rooted, IP anomaly, cross-border, e-commerce sem 3DS e alto valor.

In [ ]:
# Enriquecimento leve com KYC e merchants para sinais iniciais
m = tx.merge(kyc, on="customer_id", how="left", suffixes=("", "_kyc"))
if "merchant_id" in m.columns:
    m = m.merge(merchants, on="merchant_id", how="left", suffixes=("", "_merchant"))

signals = {}
for col, label in [
    ("sanctions_screening_hit", "Transações com sanctions hit"),
    ("ip_anomaly", "IP anomaly"),
    ("device_rooted", "Device rooted"),
    ("cross_border", "Cross-border"),
]:
    if col in m.columns:
        signals[label] = int((m[col].astype(str) == "Yes").sum())

if "sanctions_list_hit" in kyc.columns:
    signals["Clientes com sanctions list hit"] = int((kyc["sanctions_list_hit"].astype(str) == "Yes").sum())
if "pep" in kyc.columns:
    signals["Clientes PEP"] = int((kyc["pep"].astype(str) == "Yes").sum())
if "risk_rating" in kyc.columns:
    signals["Clientes KYC High Risk"] = int((kyc["risk_rating"].astype(str).str.lower() == "high").sum())
if "mcc_risk" in merchants.columns:
    signals["Merchants com MCC High Risk"] = int((merchants["mcc_risk"].astype(str).str.lower() == "high").sum())
if "merchant_high_risk_flag" in merchants.columns:
    signals["Merchants high risk flag"] = int((merchants["merchant_high_risk_flag"].astype(str) == "Yes").sum())
if "country_risk_receiver" in m.columns:
    signals["Receiver país High Risk"] = int((m["country_risk_receiver"].astype(str).str.lower() == "high").sum())
if {"transaction_type", "card_present", "auth_3ds"}.issubset(m.columns):
    signals["Card e-commerce sem 3DS"] = int(((m["transaction_type"] == "Card") & (m["card_present"].astype(str) == "No") & (m["auth_3ds"].astype(str).isin(["No", "n/a", "nan"]))).sum())
if "amount_brl" in m.columns:
    signals["Valores >= R$50k"] = int((m["amount_brl"] >= 50000).sum())

signals_df = pd.DataFrame([{"signal": k, "count": v} for k, v in signals.items()]).sort_values("count", ascending=False)
signals_df.to_csv(OUT_DIR / "04_initial_aml_signals.csv", index=False)
signals_df

In [ ]:
top = signals_df.head(12).sort_values("count")
plt.figure(figsize=(8, 5))
plt.barh(top["signal"], top["count"])
plt.title("Principais sinais AML iniciais")
plt.xlabel("Quantidade")
plt.tight_layout()
plt.savefig(OUT_DIR / "chart_06_sinais_aml_top12.png", dpi=160)
plt.show()

## 7. Conclusão da EDA

Conclusão operacional: a base é suficiente para avançar para regras e priorização. Os sinais mais relevantes serão combinados, não usados isoladamente, para reduzir falso positivo e aumentar explicabilidade.